# 07. Pipeline de Inferencia y Priorización para Analista — IBM TabFormer
**Unidad 1:** Detección de Fraude en Transacciones con Tarjeta de Crédito  
**Dataset:** Lote de Inferencia Desacoplado (`sample_inference.csv`)  
**Entorno:** Python 3.12 | scikit-learn | Polars

---

### Objetivos del Notebook:
1. Cargar el pipeline de preprocesamiento serializado (`preprocessor.joblib`) y el modelo baseline (`baseline_logistic_regression.joblib`).
2. Recibir un lote de transacciones nuevas desde `data/processed/sample_inference.csv`.
3. Validar el contrato de esquema de entrada (asegurando que **no se exija la presencia de la etiqueta `Is Fraud?`**).
4. Aplicar las transformaciones aprendidas en el preprocesador ajustado.
5. Calcular probabilidades de fraude y derivar una **Puntuación de Riesgo (Risk Score de 0 a 100)**.
6. Asignar niveles de prioridad para la revisión del analista humano (`ALTA`, `MEDIA`, `BAJA`).
7. Preservar el identificador de transacción (`transaction_id`) para garantizar la trazabilidad de cada alerta.
8. Exportar los resultados priorizados a `data/predictions/inference_results.csv`.


In [1]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR
while not (PROJECT_ROOT / "notebooks").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from notebooks._comun.config import (
    INFERENCE_SAMPLE_FILE,
    PREPROCESSOR_ARTIFACT,
    LOGISTIC_MODEL_ARTIFACT,
    INFERENCE_RESULTS_FILE,
    SELECTED_FEATURES,
)
from notebooks._comun.data_utils import load_artifact
print(f"Raíz del proyecto: {PROJECT_ROOT}")


Raíz del proyecto: /home/glender/ProyectosUNi/iaupeu


In [2]:
import polars as pl
import numpy as np
import pandas as pd

# Carga de artefactos
preprocessor = load_artifact(PREPROCESSOR_ARTIFACT)
model = load_artifact(LOGISTIC_MODEL_ARTIFACT)

print("Preprocesador y Modelo Baseline cargados exitosamente.")


Preprocesador y Modelo Baseline cargados exitosamente.


## 1. Carga del Lote de Transacciones Nuevas
Cargamos el archivo CSV de transacciones a evaluar (`sample_inference.csv`).


In [3]:
# Lectura del lote
df_new = pl.read_csv(INFERENCE_SAMPLE_FILE)
print(f"Transacciones recibidas para inferencia: {df_new.shape[0]:,} filas x {df_new.shape[1]} columnas")
print("Columnas presentes:", df_new.columns)

# Verificación de que la etiqueta no está presente ni es requerida
assert "Is Fraud?" not in df_new.columns and "is_fraud" not in df_new.columns, \
    "Error: El lote de inferencia no debe contener la etiqueta de fraude."
print("Validación exitosa: El flujo opera sin etiqueta de fraude.")


Transacciones recibidas para inferencia: 1,000 filas x 20 columnas
Columnas presentes: ['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'hour', 'minute', 'day_of_week', 'is_weekend', 'is_night', 'mcc_category', 'transaction_id']
Validación exitosa: El flujo opera sin etiqueta de fraude.


## 2. Validación de Contrato de Datos de Entrada
Verificamos que el DataFrame contenga todas las variables explicativas requeridas por el pipeline.


In [4]:
missing_features = [col for col in SELECTED_FEATURES if col not in df_new.columns]
if missing_features:
    raise ValueError(f"Faltan características obligatorias en la entrada: {missing_features}")

print(f"Todas las características obligatorias están presentes: {SELECTED_FEATURES}")


Todas las características obligatorias están presentes: ['Amount', 'hour', 'minute', 'day_of_week', 'Use Chip', 'mcc_category']


## 3. Transformación de Características y Scoring
Aplicamos `preprocessor.transform()` y calculamos las probabilidades continuas de fraude.


In [5]:
# Preparación de matriz
X_inference_df = df_new.select(SELECTED_FEATURES).to_pandas()
X_trans = preprocessor.transform(X_inference_df)

# Predicción de probabilidades (clase 1 = Fraude)
probabilities = model.predict_proba(X_trans)[:, 1]
predictions = model.predict(X_trans)

# Cálculo de Risk Score calibrado en escala 0 a 100
risk_scores = np.round(probabilities * 100, 2)

print(f"Inferencia ejecutada sobre {len(probabilities)} transacciones.")
print(f"Rango de Risk Score obtenido: {risk_scores.min():.2f} a {risk_scores.max():.2f}")


Inferencia ejecutada sobre 1000 transacciones.
Rango de Risk Score obtenido: 0.30 a 98.99


## 4. Priorización para Revisión Humana
El sistema no bloquea automáticamente todas las operaciones, sino que clasifica las alertas para la cola de trabajo del analista:
- **ALTA PRIORIDAD:** Risk Score $\ge 70$ (Revisión inmediata / retención preventiva).
- **MEDIA PRIORIDAD:** $40 \le$ Risk Score $< 70$ (Monitoreo reforzado / solicitud de factor 2FA).
- **BAJA PRIORIDAD:** Risk Score $< 40$ (Aprobación automática estándar).


In [6]:
# Crear categorías de prioridad
priority_levels = []
for score in risk_scores:
    if score >= 70.0:
        priority_levels.append("ALTA PRIORIDAD")
    elif score >= 40.0:
        priority_levels.append("MEDIA PRIORIDAD")
    else:
        priority_levels.append("BAJA PRIORIDAD")

# Construir DataFrame de resultados con trazabilidad
df_results = df_new.with_columns([
    pl.Series("fraud_probability", probabilities.round(4)),
    pl.Series("risk_score", risk_scores),
    pl.Series("prediction", predictions),
    pl.Series("priority_level", priority_levels),
])

print("Resumen de Transacciones por Nivel de Prioridad:")
print(df_results.group_by("priority_level").len().sort("len", descending=True))

print("\nTop 5 transacciones de mayor riesgo:")
top_risky = (
    df_results.sort("risk_score", descending=True)
    .select(["transaction_id", "Amount", "Use Chip", "mcc_category", "hour", "risk_score", "priority_level"])
    .head(5)
)
print(top_risky)


Resumen de Transacciones por Nivel de Prioridad:
shape: (3, 2)
┌─────────────────┬─────┐
│ priority_level  ┆ len │
│ ---             ┆ --- │
│ str             ┆ u32 │
╞═════════════════╪═════╡
│ BAJA PRIORIDAD  ┆ 623 │
│ MEDIA PRIORIDAD ┆ 280 │
│ ALTA PRIORIDAD  ┆ 97  │
└─────────────────┴─────┘

Top 5 transacciones de mayor riesgo:
shape: (5, 7)
┌────────────────┬─────────┬─────────────┬────────────────────┬──────┬────────────┬────────────────┐
│ transaction_id ┆ Amount  ┆ Use Chip    ┆ mcc_category       ┆ hour ┆ risk_score ┆ priority_level │
│ ---            ┆ ---     ┆ ---         ┆ ---                ┆ ---  ┆ ---        ┆ ---            │
│ i64            ┆ f64     ┆ str         ┆ str                ┆ i64  ┆ f64        ┆ str            │
╞════════════════╪═════════╪═════════════╪════════════════════╪══════╪════════════╪════════════════╡
│ 214            ┆ 1396.04 ┆ Chip        ┆ Airlines           ┆ 10   ┆ 98.99      ┆ ALTA PRIORIDAD │
│                ┆         ┆ Transaction ┆   

## 5. Exportación del Archivo de Predicciones
Exportamos el archivo `inference_results.csv` para la integración con sistemas operativos y dashboards de gestión de fraude.


In [7]:
# Seleccionar columnas clave y ordenar por prioridad de revisión
export_cols = [
    "transaction_id",
    "User",
    "Card",
    "Year",
    "Month",
    "Day",
    "Time",
    "Amount",
    "Use Chip",
    "mcc_category",
    "fraud_probability",
    "risk_score",
    "prediction",
    "priority_level",
]

df_export = df_results.select(export_cols).sort("risk_score", descending=True)
df_export.write_csv(INFERENCE_RESULTS_FILE)

print(f"Resultados de inferencia exportados exitosamente a: {INFERENCE_RESULTS_FILE}")
print(f"Dimensiones exportadas: {df_export.shape[0]} transacciones x {df_export.shape[1]} columnas")


Resultados de inferencia exportados exitosamente a: /home/glender/ProyectosUNi/iaupeu/data/predictions/inference_results.csv
Dimensiones exportadas: 1000 transacciones x 14 columnas
